# Análisis de Violencia Digital - Fase 1 y 2
Este cuaderno contiene las Fases 1 y 2 de nuestro Plan de Acción.
- **Fase 1:** Carga de datos y Entendimiento de escalas.
- **Fase 2:** Limpieza, Transformación (Feature Engineering) y creación de Índices.


In [4]:
%pip install pandas openpyxl
import pandas as pd
import numpy as np
import os

# Definir la ruta de los archivos (Ajusta la ruta si es necesario)
DB_DIR = './DBs'

# Archivos
file_estudiantes = os.path.join(DB_DIR, 'BBDD_Estudiantes_dash.xlsx')
file_docentes = os.path.join(DB_DIR, 'BBDD_Docentes_dash.xlsx')
file_padres = os.path.join(DB_DIR, 'BBDD_Padres_dash.xlsx')


Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 10.8 MB 2.6 MB/s eta 0:00:01
     |████████████████████████████████| 250 kB 21.0 MB/s eta 0:00:01
     |████████████████████████████████| 349 kB 53.6 MB/s eta 0:00:01
     |████████████████████████████████| 510 kB 41.1 MB/s eta 0:00:01
     |████████████████████████████████| 5.3 MB 40.5 MB/s eta 0:00:01
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


## 1. Carga de Datos
Cargamos los archivos de Excel a DataFrames de Pandas. Asegúrate de tener instalados `pandas` y `openpyxl`.
`(pip install pandas openpyxl)`


In [5]:
# Carga de las tres bases principales
try:
    df_estudiantes = pd.read_excel(file_estudiantes)
    df_docentes = pd.read_excel(file_docentes)
    df_padres = pd.read_excel(file_padres)
    print("¡Archivos cargados con éxito!")
    print(f"Estudiantes: {df_estudiantes.shape}")
    print(f"Docentes: {df_docentes.shape}")
    print(f"Padres: {df_padres.shape}")
except Exception as e:
    print("Error al cargar:", e)


¡Archivos cargados con éxito!
Estudiantes: (443, 69)
Docentes: (187, 69)
Padres: (330, 90)


## 2. Exploración y Mapeo de Escalas (Fase 1)
El primer paso es ver qué nombres de columnas tenemos. Especialmente, nos interesa localizar cuáles son las preguntas que formarán nuestros índices (ej. Q22 a Q26 para estudiantes).


In [6]:
# Muestra los nombres de las columnas para los estudiantes.
# Tienes que identificar exactamente el texto o código de la columna de las preguntas requeridas.
list(df_estudiantes.columns)


['ID_Control',
 'Marca temporal',
 'Municipio',
 'Unidad Educativa',
 'Autoidentificación cultural',
 'Si respondiste "Si" a la auto identificación cultural, indica cual',
 '¿Cual es tu idioma materno?',
 'D1. ¿Cuantos años tienes?',
 'D2. ¿Género?',
 'D3. ¿En que grado o curso estás actualmente?',
 'D4. ¿Cómo describirías el lugar donde vives?',
 '1. ¿Con qué dispositivo te conectas a internet con más frecuencia? \n(Elige una sola opción, la que uses más)',
 '2. ¿Con qué frecuencia usas internet en un día normal de semana?',
 '3. ¿Desde dónde te conectas principalmente a internet?',
 '4. ¿El dispositivo que usas principalmente para conectarte a internet es…?',
 '5. ¿Quién cubre principalmente el costo de tu conexión a internet o datos móviles?',
 '6. ¿Sientes que puedes opinar, participar o expresarte en grupos digitales escolares con la misma comodidad que tus compañeros/as de otro género?',
 '7. ¿Cuál es la red social que más usas?',
 '8. ¿Cuál es la aplicación que más usas? \n(Pued

## 3. Creación de Diccionarios de Puntuación
Antes de sumar, debemos convertir los textos de respuesta (ej: "Totalmente de acuerdo", "A veces", "Sí", "No") a números.
Dependiendo de cómo revises en los archivos Word originales (.docx), completarás estos diccionarios.


In [7]:
# EJEMPLO: Si la pregunta tiene respuestas múltiples de 1 a 5
escala_likert = {
    "1. No sé nada": 1,
    "2. Sé poco": 2,
    "3. Sé bastante": 3,
    "4. Lo conozco bien": 4,
}

# EJEMPLO: Si la pregunta es binaria
escala_binaria = {
    "Sí": 1,
    "No": 0
}

# Una vez identifiquemos los valores reales, aplicaremos estos mapeos a los dataframes


## 4. Feature Engineering y Creación de Índices (Fase 2)
Aquí ejecutaremos el cálculo final para los estudiantes. Sustituye `['P22', 'P23', ...]` por los nombres reales de las columnas que arroje el paso de exploración.


In [10]:
# --- ESTUDIANTES: CÁLCULO DE ÍNDICE DE CONOCIMIENTO ---

def evaluar_conocimiento(df):
    # 1. Definir los nombres exactos de tus columnas (tal como me las enviaste)
    col_q22 = '22. ¿Qué es el Grooming?'
    col_q23 = '23. ¿Qué es la Sextorsión?'
    col_q24 = '24. ¿Cuál de estos comportamientos consideras que es violencia sexual en línea?'
    col_q25 = '25. Cuál de estas acciones es considerada ciberacoso o "cyberbullying"?'
    col_q26 = '26. ¿Cuál de estas situaciones consideras una forma de violencia digital, aunque no sea de tipo sexual?\n(Puede marcar más de una opción)'
    
    # 2. Funciones de calificación (1 punto si es correcto, 0 si no)
    # Convertimos a minúsculas (.lower()) para evitar errores de mayúsculas/minúsculas en Excel.
    
    # Q22: Grooming (Correcta: adulto contacta)
    df['Puntaje_Q22'] = df[col_q22].apply(lambda x: 1 if 'adulto contacta' in str(x).lower() else 0)
    
    # Q23: Sextorsión (Correcta: amenaza, imágenes íntimas)
    df['Puntaje_Q23'] = df[col_q23].apply(lambda x: 1 if 'amenaza' in str(x).lower() or 'íntimas' in str(x).lower() else 0)
    
    # Q24: Violencia sexual en línea (Correcta: sin consentimiento)
    df['Puntaje_Q24'] = df[col_q24].apply(lambda x: 1 if 'sin consentimiento' in str(x).lower() else 0)
    
    # Q25: Ciberacoso (Correcta: mensajes ofensivos o burlas)
    df['Puntaje_Q25'] = df[col_q25].apply(lambda x: 1 if 'ofensivos' in str(x).lower() or 'burlas' in str(x).lower() else 0)
    
    # Q26: Situaciones de violencia (Múltiples opciones, "Todas las anteriores" o cualquiera combinada es correcta)
    def puntaje_q26(texto):
        t = str(texto).lower()
        if 'todas las anteriores' in t: return 1
        if 'suplantación' in t or 'excluir' in t or 'falsa' in t: return 1
        return 0
        
    df['Puntaje_Q26'] = df[col_q26].apply(puntaje_q26)
    
    # 3. CREAR EL ÍNDICE FINAL SUMANDO LOS PUNTAJES (Nota sobre 5)
    df['Indice_Conocimiento'] = df[['Puntaje_Q22', 'Puntaje_Q23', 'Puntaje_Q24', 'Puntaje_Q25', 'Puntaje_Q26']].sum(axis=1)
    
    return df

# Aplicamos la evaluación a la base de estudiantes
df_estudiantes = evaluar_conocimiento(df_estudiantes)

# Observar el resultado de los primeros 10 estudiantes
display(df_estudiantes[['Indice_Conocimiento', 'Puntaje_Q22','Puntaje_Q23','Puntaje_Q24', 'Puntaje_Q25', 'Puntaje_Q26']].head(10))



,Indice_Conocimiento,Puntaje_Q22,Puntaje_Q23,Puntaje_Q24,Puntaje_Q25,Puntaje_Q26
0,4,1,1,0,1,1
1,4,0,1,1,1,1
2,5,1,1,1,1,1
3,5,1,1,1,1,1
4,5,1,1,1,1,1
5,3,0,0,1,1,1
6,5,1,1,1,1,1
7,2,0,0,0,1,1
8,5,1,1,1,1,1
9,5,1,1,1,1,1


In [12]:
# --- ESTUDIANTES: CÁLCULO DE ÍNDICE MÁXIMO DE PREVENCIÓN ---

def calcular_indice_prevencion_final(df):
    # --- PARTE 1: COLUMNAS DE COMPORTAMIENTO (SINGLE CHOICE) ---
    col_q14 = '14. Cuando juegas en línea, ¿con quién juegas habitualmente?' 
    col_q15 = '15. Cuando juegas con personas desconocidas en línea, ¿qué tipo de información has compartido con ellas?\n(Puede marcar más de una opción)'
    col_q20 = '20. ¿Alguna vez sentiste presión para enviar fotos personales o íntimas a alguien por internet?'
    col_q28 = '28. Si alguien que no conoces te pide fotos personales por internet, ¿qué harías?'
    col_q29 = '29. ¿Qué haces normalmente con tu contraseña de redes sociales o correo electrónico?'
    col_q30 = '30. ¿Con qué frecuencia ajustas la configuración de privacidad de tus redes sociales?'
    col_q31 = '31. Antes de publicar una foto o video en redes sociales, ¿qué haces habitualmente?'
    
    df['Prev_Q14'] = df[col_q14].apply(lambda x: 1 if 'solo' in str(x).lower() or 'conozco en persona' in str(x).lower() else 0)
    df['Prev_Q15'] = df[col_q15].apply(lambda x: 1 if 'nunca comparto' in str(x).lower() else 0)
    df['Prev_Q20'] = df[col_q20].apply(lambda x: 1 if 'nunca' in str(x).lower() or 'me negué' in str(x).lower() else 0)
    df['Prev_Q28'] = df[col_q28].apply(lambda x: 1 if 'negarme' in str(x).lower() or 'bloquear' in str(x).lower() else 0)
    df['Prev_Q29'] = df[col_q29].apply(lambda x: 1 if 'nadie' in str(x).lower() or 'padres' in str(x).lower() else 0)
    df['Prev_Q30'] = df[col_q30].apply(lambda x: 1 if 'privado' in str(x).lower() or 'a veces' in str(x).lower() else 0)
    df['Prev_Q31'] = df[col_q31].apply(lambda x: 1 if 'aseguro' in str(x).lower() or 'nunca publico' in str(x).lower() else 0)
    
    df['Puntaje_Comportamiento'] = df[['Prev_Q14', 'Prev_Q15', 'Prev_Q20', 'Prev_Q28', 'Prev_Q29', 'Prev_Q30', 'Prev_Q31']].sum(axis=1)

    # --- PARTE 2: COLUMNAS DE LA MATRIZ Q19 (NIVEL DE CONOCIMIENTO 1-4) ---
    cols_q19 = [
        '19. ¿Qué tanto conoces sobre los siguientes temas para protegerte en internet? \nMarca tu nivel de conocimiento: [PRIVACIDAD DIGITAL (proteger tus datos y fotos personales)]',
        '19. ¿Qué tanto conoces sobre los siguientes temas para protegerte en internet? \nMarca tu nivel de conocimiento: [CIBERACOSO (acoso o burlas repetidas por internet)]',
        '19. ¿Qué tanto conoces sobre los siguientes temas para protegerte en internet? \nMarca tu nivel de conocimiento: [SEGURIDAD DIGITAL (contraseñas, configuración de privacidad)]',
        '19. ¿Qué tanto conoces sobre los siguientes temas para protegerte en internet? \nMarca tu nivel de conocimiento: [DERECHOS DIGITALES (tus derechos como usuario/a en internet)]',
        '19. ¿Qué tanto conoces sobre los siguientes temas para protegerte en internet? \nMarca tu nivel de conocimiento: [USO RESPONSABLE DE INTERNET Y REDES SOCIALES]'
    ]
    
    # Extracción numérica de Q19
    for col in cols_q19:
        df[col + '_num'] = df[col].astype(str).str.extract(r'(\d+)')[0].astype(float).fillna(0)
        
    df['Puntaje_Q19'] = df[[col + '_num' for col in cols_q19]].sum(axis=1)

    # --- PARTE 3: MATRIZ Q27 (NIVEL DE CONOCIMIENTO EN TEXTO) ---
    cols_q27 = [
        '27. ¿Qué tanto conoces sobre los siguientes riesgos del uso de la Inteligencia Artificial (IA) en internet?\nMarca tu nivel de conocimiento: [DEEPFAKES (videos o fotos falsas creadas con IA para hacerse pasar por alguien)]',
        '27. ¿Qué tanto conoces sobre los siguientes riesgos del uso de la Inteligencia Artificial (IA) en internet?\nMarca tu nivel de conocimiento: [VOZ ARTIFICIAL (audios falsos generados con IA imitando la voz de una persona)]',
        '27. ¿Qué tanto conoces sobre los siguientes riesgos del uso de la Inteligencia Artificial (IA) en internet?\nMarca tu nivel de conocimiento: [IMÁGENES ÍNTIMAS FALSAS GENERADAS CON IA SIN CONSENTIMIENTO DE LA PERSONA]',
        '27. ¿Qué tanto conoces sobre los siguientes riesgos del uso de la Inteligencia Artificial (IA) en internet?\nMarca tu nivel de conocimiento: [CHATBOTS QUE SE HACEN PASAR POR PERSONAS REALES PARA ENGAÑAR O MANIPULAR]',
        '27. ¿Qué tanto conoces sobre los siguientes riesgos del uso de la Inteligencia Artificial (IA) en internet?\nMarca tu nivel de conocimiento: [USO DE LA IA PARA ROBAR O SUPLANTAR IDENTIDADES DIGITAL]'
    ]

    def convertir_escala_conocimiento(texto):
        t = str(texto).lower()
        if 'nada' in t: return 1
        if 'poco' in t: return 2
        if 'bastante' in t: return 3
        if 'bien' in t: return 4
        return 0
        
    for col in cols_q27:
        df[col + '_num'] = df[col].apply(convertir_escala_conocimiento)
        
    df['Puntaje_Q27'] = df[[col + '_num' for col in cols_q27]].sum(axis=1)

    # --- ÍNDICE FINAL (SUMA TOTAL DE CAPACIDAD DE PREVENCIÓN) ---
    df['Indice_Prevencion'] = df['Puntaje_Comportamiento'] + df['Puntaje_Q19'] + df['Puntaje_Q27']
    
    return df

# Ejecución
df_estudiantes = calcular_indice_prevencion_final(df_estudiantes)

print("¡Cálculo Exitoso! Muestra del Índice de Capacidad de Prevención:")
display(df_estudiantes[['Indice_Prevencion', 'Puntaje_Comportamiento', 'Puntaje_Q19', 'Puntaje_Q27']].head(10))


¡Cálculo Exitoso! Muestra del Índice de Capacidad de Prevención:


,Indice_Prevencion,Puntaje_Comportamiento,Puntaje_Q19,Puntaje_Q27
0,36.0,7,14.0,15
1,46.0,6,20.0,20
2,46.0,6,20.0,20
3,36.0,7,14.0,15
4,25.0,4,15.0,6
5,31.0,7,13.0,11
6,39.0,3,16.0,20
7,35.0,6,14.0,15
8,44.0,7,20.0,17
9,47.0,7,20.0,20
